# BirdCLEF 2024: Full 182-Species Embedding Extraction
This notebook extracts Perch v2 embeddings for all 182 species in the competition.
Estimated runtime: ~45-60 minutes on T4 GPU.

In [ ]:
# Mandatory setup for Perch and BirdNET to work correctly on Kaggle
!pip install -q --upgrade "tensorflow[and-cuda]>=2.16.1" birdnetlib pyarrow scikit-learn resampy

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import librosa
from pathlib import Path
from tqdm.auto import tqdm

# --- Configuration ---
INPUT_DIR = Path('/kaggle/input')
AUDIO_DIR = INPUT_DIR / 'competitions/birdclef-2024/train_audio'
OUTPUT_DIR = Path('/kaggle/working/cache/embeddings')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SR = 32000
CHUNK_SEC = 5.0

# --- Model Loading ---
import tensorflow as tf
from pathlib import Path
import numpy as np
import librosa

import tensorflow as tf
import numpy as np

MODEL_PATH = "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2"

try:
    print(f"Loading Perch v2...")
    model = tf.saved_model.load(MODEL_PATH)
    
    # Perch v2 on Kaggle usually uses 'serving_default' as the signature name
    if 'serving_default' in model.signatures:
        perch_fn = model.signatures['serving_default']
        print("Success: Using 'serving_default' signature.")
    else:
        # Fallback: Use the first available signature
        first_sig = list(model.signatures.keys())[0]
        perch_fn = model.signatures[first_sig]
        print(f"Success: Using '{first_sig}' signature.")
        
except Exception as e:
    print(f"Failed to load model: {e}")
    perch_fn = None

def extract_perch(audio_5s_32k):
    if perch_fn is None: raise RuntimeError("Model not loaded.")
    
    # Input must be [batch, samples]
    x = tf.constant(audio_5s_32k[None, :], dtype=tf.float32)
    
    # Signature calls usually require 'inputs=' or 'input_node='
    # Most Perch models use 'inputs'
    try:
        outputs = perch_fn(inputs=x)
    except TypeError:
        outputs = perch_fn(x)
        
    # Perch v2 returns a dict with 'embedding' and 'label'
    return outputs['embedding'].numpy()[0]

# --- Metadata Setup ---
train_df = pd.read_csv(INPUT_DIR / 'competitions/birdclef-2024/train_metadata.csv')
# Subsample to 100 files per species for a balanced but fast run
sampled_df = train_df.groupby('primary_label').head(100).reset_index(drop=True)
print(f"Processing {len(sampled_df)} files across {train_df['primary_label'].nunique()} species.")

In [ ]:
from tqdm.notebook import tqdm
# 1. Verify there is data
print(f"Total files to process: {len(sampled_df)}")
results = []
for idx, row in tqdm(sampled_df.iterrows(), total=len(sampled_df)):
    path = AUDIO_DIR / row['filename']
    if not path.exists(): continue
    
    try:
        # Load only the first 5 seconds to be lightning fast
        y, _ = librosa.load(path, sr=SR, duration=CHUNK_SEC, mono=True)
        if len(y) < SR * CHUNK_SEC:
            y = np.pad(y, (0, int(SR * CHUNK_SEC) - len(y)))
        
        emb = extract_perch(y)
        results.append({
            'file_id': row['filename'],
            'species_code': row['primary_label'],
            'emb': emb
        })
        
        # Checkpoint every 1000 files
        if (idx + 1) % 1000 == 0:
            pd.DataFrame(results).to_parquet(OUTPUT_DIR / f'perch_train_182_{idx}.parquet')
            results = []
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")

# Final Save
if results:
    pd.DataFrame(results).to_parquet(OUTPUT_DIR / 'perch_train_182_final.parquet')
print("Extraction Complete!")